# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ModuleNotFoundError: No module named 'matplotlib'

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [30]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    r"C:\Users\HP\Downloads\AviationData (1).csv",
    encoding="cp1252",
    low_memory=False
)

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [31]:
#Inspecting data
print("\n" + "=" * 60)
print("DATA INSPECTION")
print("=" * 60)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values summary:")
missing_counts = df.isnull().sum().sort_values(ascending=False)
print(missing_counts)

print("\nSummary statistics for numeric columns:")
print(df.describe())


DATA INSPECTION

Column names:
['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Publication.Date']

Data types:
Event.Id                      str
Investigation.Type            str
Accident.Number               str
Event.Date                    str
Location                      str
Country                       str
Latitude                      str
Longitude                     str
Airport.Code                  str
Airport.Name                  str
Injury.Severity               str
Aircraft.damage               str
Airc

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [32]:
import re

# Fatal/serious injuries.

inj_cols = ["total_fatal_injuries", "total_serious_injuries",
            "total_minor_injuries", "total_uninjured"]

for col in inj_cols:
    for col in inj_cols:
        # if exact column exists, convert directly
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            continue

        # normalize names (remove non-alnum, lowercase) and try to find a match
        target_norm = re.sub(r'\W+', '', col).lower()
        matches = [c for c in df.columns if re.sub(r'\W+', '', str(c)).lower() == target_norm]

        if matches:
            actual = matches[0]
            df[actual] = pd.to_numeric(df[actual], errors="coerce")
            # create standardized column for downstream code
            df[col] = df[actual]
            continue

        # fallback: try Title.Case with dots (e.g., total_fatal_injuries -> Total.Fatal.Injuries)
        alt = '.'.join([p.capitalize() for p in col.split('_')])
        if alt in df.columns:
            df[alt] = pd.to_numeric(df[alt], errors="coerce")
            df[col] = df[alt]
        else:
            # if no matching column, create as NaN to avoid KeyError later
            df[col] = np.nan

print(df[inj_cols].dtypes)

print("\nMissingness in injury count columns")
print(df[inj_cols].isna().mean().round(3)*100)


total_fatal_injuries      float64
total_serious_injuries    float64
total_minor_injuries      float64
total_uninjured           float64
dtype: object

Missingness in injury count columns
total_fatal_injuries      12.8
total_serious_injuries    14.1
total_minor_injuries      13.4
total_uninjured            6.7
dtype: float64


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [33]:
import re

print("\naircraft_damage value counts (raw):")
# robustly find the aircraft damage column (normalize names)
col_name = None
for c in df.columns:
    if re.sub(r'\W+', '', str(c)).lower() == 'aircraftdamage':
        col_name = c
        break

if col_name:
    print(df[col_name].value_counts(dropna=False))
else:
    print("No 'aircraft_damage' column found. Available similar columns:",
          [c for c in df.columns if 'aircraft' in str(c).lower()])


aircraft_damage value counts (raw):
Aircraft.damage
Substantial    64148
Destroyed      18623
NaN             3194
Minor           2805
Unknown          119
Name: count, dtype: int64


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [34]:
import re

def clean_make(m):
    if pd.isna(m):
        return m
    s = str(m).upper().strip()
    
    if 'STEARMAN' in s:
        return 'STEARMAN'
    s = re.sub(r'[.,]', '', s)                                  
    s = re.sub(r'\b(INC|CORP|CORPORATION|CO|COMPANY|LLC|LTD)\b', '', s) 
    s = re.sub(r'\s+', ' ', s).strip()
    return s

df['make_u'] = df['Make'].apply(clean_make)
df['make_stage1'] = df['make_u'].apply(clean_make)


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [35]:
#Inspecting data
print("\n" + "=" * 60)
print("Data Inspection")
print("=" * 60)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values summary:")
missing_counts = df.isnull().sum().sort_values(ascending=False)
print(missing_counts)


print("\nSummary statistics for numeric columns:")
print(df.describe())


Data Inspection

Column names:
['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code', 'Airport.Name', 'Injury.Severity', 'Aircraft.damage', 'Aircraft.Category', 'Registration.Number', 'Make', 'Model', 'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description', 'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured', 'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status', 'Publication.Date', 'total_fatal_injuries', 'total_serious_injuries', 'total_minor_injuries', 'total_uninjured', 'make_u', 'make_stage1']

Data types:
Event.Id                      str
Investigation.Type            str
Accident.Number               str
Event.Date                    str
Location                      str
Country                       str
Latitude                      str
Longitude                     str
Airport.Code          

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [36]:
#Cleaning other columns
print("\n" + "=" * 60)
print("Cleaning other columns")
print("=" * 60)

#Engine type
engine_col = None
possible_engine = ['eng_type', 'Engine.Type', 'Engine Type','EngineType']
for col in possible_engine:
    if col in df.columns:
        engine_col = col
        break
if engine_col:
    print(f"\nCleaning '{engine_col}':")
    df[engine_col] = df[engine_col].astype(str).str.upper().str.strip()
    print(f"Unique values: {df[engine_col].nunique()}")
    print(f"Top 5 values:\n{df[engine_col].value_counts().head(5)}")


#Weather condition
weather_col = None
possible_weather = ['wx_cond_basic', 'Weather.Condition', 'Weather Condition', 'WeatherCondition']
for col in possible_weather:
    if col in df.columns:
        weather_col = col
        break
if weather_col:
    print(f"\nCleaning '{weather_col}':")
    df[weather_col] = df[weather_col].astype(str).str.upper().str.strip()
    print(f"Unique values: {df[weather_col].nunique()}")
    print(f"Top 5 values:\n{df[weather_col].value_counts().head(5)}")

#Number of engines
engine_col = None
possible_engines = ['num_eng', 'Number.of.Engines', 'Number of Engines', 'NumberofEngines']
for col in possible_engines:
    if col in df.columns:
        engine_col = col
        break
    
if engine_col:
    print(f"\nCleaning '{engine_col}':")


    #Convert to numeric, coercing errors to NaN, then drop NaNs
    df[engine_col] = pd.to_numeric(df[engine_col], errors = 'coerce')
    initial_shape = df.shape
    df = df.dropna(subset=[engine_col]).copy()
    print(f"After dropping NaN from '{engine_col}': {df.shape}")

    #Checking for unreasonable values
    if df[engine_col].dtype in ['int64', 'float64']:
     print(f"Min: {df[engine_col].min()}, Max: {df[engine_col].max()}")

    #Removing unrealisic values
    df = df[df[engine_col]<=8].copy()
    print(f"After removing >8 engines: {df.shape}")
else:
    print("No number of engine column found!")




Cleaning other columns

Cleaning 'Engine.Type':
Unique values: 12
Top 5 values:
Engine.Type
RECIPROCATING    69530
TURBO SHAFT       3609
TURBO PROP        3391
TURBO FAN         2481
UNKNOWN           2051
Name: count, dtype: int64

Cleaning 'Weather.Condition':
Unique values: 3
Top 5 values:
Weather.Condition
VMC    77303
IMC     5976
UNK     1118
Name: count, dtype: int64

Cleaning 'Number.of.Engines':
After dropping NaN from 'Number.of.Engines': (82805, 37)
Min: 0.0, Max: 8.0
After removing >8 engines: (82805, 37)


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [37]:
print("\n" + "=" * 60)
print("Removing columns with excess nulls")
print("=" * 60)

#Checking total rows before removing the columns
total_rows = len(df)
print(f"Total rows: {total_rows}")


Removing columns with excess nulls
Total rows: 82805


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [38]:
df.to_csv('Aviation_Accidents.csv', index=False)